## Hospital Data Analysis 

In [1]:
import pandas as pd
import sqlite3

In [2]:
df = pd.read_csv("hospital_data_analysis(in).csv", sep=";")
df.head()

,Patient_ID,Age,Gender,Condition,Procedure,Cost,Length_of_Stay,Readmission,Outcome,Satisfaction
0,1,45,Female,Heart Disease,Angioplasty,15000,5,No,Recovered,4
1,2,60,Male,Diabetes,Insulin Therapy,2000,3,Yes,Stable,3
2,3,32,Female,Fractured Arm,X-Ray and Splint,500,1,No,Recovered,5
3,4,75,Male,Stroke,CT Scan and Medication,10000,7,Yes,Stable,2
4,5,50,Female,Cancer,Surgery and Chemotherapy,25000,10,No,Recovered,4


In [3]:
conn = sqlite3.connect("db.hospital")
df.to_sql("patients", conn, if_exists="replace", index=False)

984

#### General Patient Information

In [4]:
query1 = """
SELECT COUNT(*) AS total_patients, AVG(Age) AS avg_age
FROM patients
"""
df_general = pd.read_sql_query(query1, conn)
print("General Patient Information")
print(df_general)

General Patient Information
   total_patients    avg_age
0             984  53.754065


In [5]:
df_general.to_csv("general_info.csv", index=False)

#### Overall Average Cost

In [6]:
query2 = """ 
SELECT AVG(Cost) AS overall_avg_cost
FROM patients
""" 
df_oacost = pd.read_sql_query(query2, conn)
print("Overall Average Cost")
print(df_oacost)

Overall Average Cost
   overall_avg_cost
0       8367.479675


In [7]:
df_oacost.to_csv("overall_avg_cost.csv", index=False)

#### Average Cost per Procedure

In [8]:
query3 = """
SELECT Procedure, AVG(Cost) AS avg_cost, COUNT(*) AS num_patients
FROM patients
GROUP BY procedure
ORDER BY avg_cost DESC
"""
df_cost = pd.read_sql_query(query3, conn)
print("Average Cost per Procedure")
print(df_cost)

Average Cost per Procedure
                               Procedure  avg_cost  num_patients
0               Surgery and Chemotherapy   25000.0            66
1                      Radiation Therapy   20000.0            65
2                Cardiac Catheterization   18000.0            67
3                            Angioplasty   15000.0            65
4            Delivery and Postnatal Care   12000.0            65
5                 CT Scan and Medication   10000.0            66
6                           Appendectomy    8000.0            66
7                            Lithotripsy    6000.0            65
8   Physical Therapy and Pain Management    4000.0            64
9              Cast and Physical Therapy    3000.0            67
10                       Insulin Therapy    2000.0            65
11             Medication and Counseling    1000.0            66
12                  Antibiotics and Rest     800.0            65
13                      X-Ray and Splint     500.0            6

In [9]:
df_cost.to_csv("procedure_cost.csv", index=False)

#### Overall Avg Length of Stay

In [10]:
query4 = """ 
SELECT AVG(Length_of_Stay) AS overall_length_stay
FROM patients
""" 
df_oalength = pd.read_sql_query(query4, conn)
print("Overall Avg Length of Stay")
print(df_oalength)

Overall Avg Length of Stay
   overall_length_stay
0            37.663618


In [11]:
df_oalength.to_csv("overall_length_stay.csv", index=False)

#### Length of Stay per Procedure (Avg & Max)

In [12]:
query5 = """
SELECT Procedure, AVG(Length_of_Stay) AS avg_stay, MAX(Length_of_Stay) AS max_stay
FROM patients
GROUP BY Procedure
ORDER BY avg_stay DESC
"""
df_los = pd.read_sql_query(query5,conn)
print("Length of Stay Distribution")
print(df_los)

Length of Stay Distribution
                               Procedure   avg_stay  max_stay
0               Surgery and Chemotherapy  42.651515        76
1                      Radiation Therapy  41.584615        74
2                Cardiac Catheterization  41.000000        74
3                 CT Scan and Medication  40.333333        74
4              Cast and Physical Therapy  39.000000        72
5                            Angioplasty  38.215385        71
6   Physical Therapy and Pain Management  37.859375        70
7                           Appendectomy  37.439394        72
8                            Lithotripsy  36.461538        69
9                        Insulin Therapy  36.092308        69
10           Delivery and Postnatal Care  35.523077        68
11             Medication and Counseling  35.515152        70
12                  Antibiotics and Rest  34.661538        68
13                      X-Ray and Splint  34.257576        67
14                 Epinephrine Injection  

In [13]:
df_los.to_csv("length_of_stay.csv", index=False)

#### Readmission Rate by Procedure

In [14]:
query6 = """
SELECT Procedure, SUM(CASE WHEN Readmission='Yes' THEN 1 ELSE 0 END)*1.0/COUNT(*) AS readmission_rate
FROM patients
GROUP BY Procedure
ORDER BY readmission_rate DESC
"""
df_readmission = pd.read_sql_query(query6,conn)

df_readmission['readmission_rate_percent'] = df_readmission['readmission_rate'] * 100
df_readmission['readmission_rate_percent'] = df_readmission['readmission_rate_percent'].round(1)

df_readmission = df_readmission[['Procedure', 'readmission_rate_percent']]
df_readmission.rename(columns={'readmission_rate_percent': 'Readmission Rate (%)'}, inplace=True)

print("Readmission Rate by Procedure")
print(df_readmission)

Readmission Rate by Procedure
                               Procedure  Readmission Rate (%)
0                Cardiac Catheterization                 100.0
1                            Angioplasty                  98.5
2                       X-Ray and Splint                  50.0
3               Surgery and Chemotherapy                  50.0
4                 CT Scan and Medication                  50.0
5                           Appendectomy                  50.0
6                        Insulin Therapy                   1.5
7                      Radiation Therapy                   0.0
8   Physical Therapy and Pain Management                   0.0
9              Medication and Counseling                   0.0
10                           Lithotripsy                   0.0
11                 Epinephrine Injection                   0.0
12           Delivery and Postnatal Care                   0.0
13             Cast and Physical Therapy                   0.0
14                  Antib

In [15]:
df_readmission.to_csv("readmission_rate.csv", index=False)

#### Overall Avg Satisfaction

In [16]:
query7 = """ 
SELECT AVG(Satisfaction) AS overall_avg_satisfaction
FROM patients
""" 
df_oasat = pd.read_sql_query(query7, conn)
print("Overall Avg Satisfaction")
print(df_oasat)

Overall Avg Satisfaction
   overall_avg_satisfaction
0                  3.598577


In [17]:
df_oasat.to_csv("overall_avg_sat.csv", index=False)

#### Outcome & Satisfaction

In [18]:
query8 = """
SELECT Outcome, AVG(Satisfaction) AS avg_satisfaction, COUNT(*) AS count
FROM patients
GROUP BY Outcome
ORDER BY avg_satisfaction DESC 
"""
df_outcome = pd.read_sql_query(query8, conn)
print("Outcome & Satisfaction")
print(df_outcome)

Outcome & Satisfaction
     Outcome  avg_satisfaction  count
0  Recovered          3.835871    591
1     Stable          3.241730    393


In [19]:
df_outcome.to_csv("outcome_satisfaction.csv", index=False)

#### Comparison by Gender

In [20]:
query9 = """ 
SELECT Gender, AVG(Cost) AS avg_cost, AVG(Length_of_Stay) AS avg_stay
FROM patients
GROUP BY Gender
"""
df_gender = pd.read_sql_query(query9,conn)
print("Comparison by Gender")
print(df_gender)

Comparison by Gender
   Gender      avg_cost   avg_stay
0  Female  10458.015267  37.715649
1    Male   5986.086957  37.604348


In [21]:
df_gender.to_csv("gender_comparison.csv", index=False)